In [3]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

# ---------------------------
# 1. Load Data
# ---------------------------
sales = pd.read_csv("sales.csv")
models = pd.read_csv("mobilemodels.csv")

# Ensure correct column names
sales.columns = sales.columns.str.lower()
models.columns = models.columns.str.lower()

# ---------------------------
# 2. Merge model names
# ---------------------------
df = sales.merge(models[["model_id", "model_name"]], on="model_id", how="left")

# Drop transactions without model names
df = df.dropna(subset=["model_name"])

# ---------------------------
# 3. Expand quantity into separate rows
# ---------------------------
df["quantity"] = df["quantity"].fillna(1).astype(int)

expanded_rows = []
for _, row in df.iterrows():
    expanded_rows.extend([row["model_name"]] * row["quantity"])

# Create a new dataframe with transaction IDs repeated properly
expanded_df = df.loc[df.index.repeat(df["quantity"])].reset_index(drop=True)

# ---------------------------
# 4. Create Basket Format
# ---------------------------
basket = (
    expanded_df.groupby(["sale_id", "model_name"])["model_name"]
    .count().unstack().fillna(0)
)

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

# Remove empty rows
basket = basket.loc[(basket.sum(axis=1) > 0)]

# ---------------------------
# 5. Apriori Algorithm
# ---------------------------
frequent_items = apriori(basket, 
                         min_support=0.01, 
                         use_colnames=True)

print("Frequent Itemsets:")
display(frequent_items.head(10))

# ---------------------------
# 6. Association Rules
# ---------------------------
rules = association_rules(frequent_items, 
                          metric="lift", 
                          min_threshold=1)

rules = rules.sort_values("lift", ascending=False)

print("Top 10 Rules:")
display(rules.head(10))


C:\Users\HP\AppData\Local\Temp\ipykernel_8240\3665306382.py:42: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


Frequent Itemsets:


C:\Users\HP\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.039350,(Series1)
1,0.051375,(Series10)
2,0.039950,(Series11)
3,0.044000,(Series12)
4,0.044675,(Series13)
5,0.073875,(Series14)
6,0.058975,(Series15)
7,0.065000,(Series16)
8,0.035575,(Series17)
9,0.076250,(Series18)


Top 10 Rules:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
